# 01 — Your First CGE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/01_your_first_cge.ipynb)

**Model:** Hosoe, Gasawa & Hashimoto's pedagogical `splcge` model.

```text
Capital ─┐              ┌─ Bread ─┐
         ├─ production ─┤         │
Labor ───┘              └─ Milk ──┼─→ Household consumption
                                  │
Household owns Capital + Labor ───┘
```

There is **no government, no trade, no tax system, and no intermediate-input network**.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

WORKSPACE = Path("/content") if Path("/content").exists() else Path.home() / ".cache"
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORKSPACE / "CGE-core-colab"

if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main", "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/miraflor/CGE-core.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core
print("✓ CGE-Core", cge_core.__version__)
print("✓ Repository:", REPO_DIR)


subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "amplpy.modules", "install", "coin"],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

module_path = subprocess.check_output(
    [sys.executable, "-m", "amplpy.modules", "path"],
    text=True,
).strip()
os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found after installing the COIN module."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)

## 2. Load the benchmark economy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pyomo.environ import value

from cge_core import PyCGE, example_data
from cge_core.examples.splcge_model_def import SplModelDef

data_dir = example_data("splcge")
sam = pd.read_csv(data_dir / "param-sam-.csv", index_col=0)
display(sam)

## 3. Calibrate the baseline

In [ ]:
cge = PyCGE(SplModelDef())
cge.model_data(data_dir)
cge.model_instance("pf", "LAB")
cge.model_drop_redundant("eqpf", "LAB")
cge.model_calibrate(SOLVER)

baseline = pd.DataFrame({
    "good": list(cge.base.i),
    "output_Z": [value(cge.base.Z[i]) for i in cge.base.i],
    "consumption_X": [value(cge.base.X[i]) for i in cge.base.i],
    "goods_price_px": [value(cge.base.px[i]) for i in cge.base.i],
})
display(baseline)

print("Factor prices:")
for h in cge.base.h:
    print(f"  {h}: {value(cge.base.pf[h]):.4f}")
print("Baseline welfare objective:", value(cge.base.obj))

## 4. Your first shock 👇

Give the household more of one factor. The default is **10% more capital**.

In [ ]:
# 👇 EDIT THESE
FACTOR = "CAP"          # "CAP" or "LAB"
FACTOR_CHANGE_PCT = 10

## 5. Solve the counterfactual economy

In [ ]:
if FACTOR not in list(cge.base.h):
    raise ValueError(f"FACTOR must be one of {list(cge.base.h)}")

base_endowment = value(cge.base.FF[FACTOR])
new_endowment = base_endowment * (1 + FACTOR_CHANGE_PCT / 100)

cge.model_sim()
cge.model_modify_sim("FF", FACTOR, new_endowment)
cge.model_solve(SOLVER)
results = cge.model_compare()

print(f"{FACTOR} endowment: {base_endowment:.4f} → {new_endowment:.4f}")
print(f"Welfare: {value(cge.base.obj):.4f} → {value(cge.sim.obj):.4f}")

## 6. See the equilibrium response

In [ ]:
headline = results[results["component"].isin(["Z", "X", "pf", "px"])].copy()
display(
    headline[["component", "index_1", "base_value", "sim_value", "difference", "pct_change"]]
    .style.format({
        "base_value": "{:.4f}",
        "sim_value": "{:.4f}",
        "difference": "{:+.4f}",
        "pct_change": "{:+.2f}%",
    })
)

In [ ]:
for component, title in [
    ("Z", "Output"),
    ("X", "Household consumption"),
    ("pf", "Factor prices"),
]:
    part = results[results["component"] == component]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(part["index_1"].astype(str), part["pct_change"].astype(float))
    ax.axhline(0, linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel("% change from baseline")
    plt.show()

## What you learned

You changed one exogenous resource endowment. The model then found a new equilibrium in which firms optimize, the household optimizes, and goods and factor markets clear simultaneously.

## Next

Notebook 02 adds government, taxes, imports, exports, saving, and investment.

[Open Notebook 02 in Colab](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/02_open_economy_cge.ipynb)